# CUIMC Appointment Simulation Analysis

This notebook mirrors the repository scripts:

- `scripts/run_simulation.py` loads `configs/baseline.yaml` and runs one simulation.
- `scripts/compare_scenarios.py` compares `configs/baseline.yaml` and `configs/scenario_2.yaml`.

The simulation is a day-level rolling appointment calendar with Poisson arrivals, FCFS earliest-open-day booking, delay-dependent balking and no-shows, future-day cancellations, burn-in, measurement, and cooldown periods.

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display


def find_repo_dir(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "baseline.yaml").exists() and (
            candidate / "simulation" / "config_loader.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows, result_metrics_from_result
from analysis.plot_style import (
    ARRIVAL_COLOR,
    BALKING_COLOR,
    BASELINE_COLOR,
    CANCELLATION_COLOR,
    NO_SHOW_COLOR,
)
from simulation.config_loader import load_config
from simulation.engine import ClinicAppointmentSimulation

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

In [ ]:
if not (REPO_DIR / "configs" / "baseline.yaml").exists():
    raise FileNotFoundError("Could not find configs/baseline.yaml from the current notebook location.")

CONFIG_PATHS = {
    "Baseline": REPO_DIR / "configs" / "baseline.yaml",
    "Scenario 2": REPO_DIR / "configs" / "scenario_2.yaml",
}

CONFIG_PATHS

## Configuration

The YAML files define the booking capacity, calendar horizon, simulation periods, random seed, and class-specific arrival and behavior parameters.

In [ ]:
def describe_rule(rule):
    if all(hasattr(rule, attr) for attr in ("threshold", "low", "high")):
        return f"tau <= {rule.threshold}: {rule.low:.2f}; tau > {rule.threshold}: {rule.high:.2f}"
    return rule.__class__.__name__

configs = {name: load_config(path) for name, path in CONFIG_PATHS.items()}

global_config_df = pd.DataFrame(
    [
        {
            "scenario": name,
            "slots_per_day": config.slots_per_day,
            "horizon_days": config.horizon_days,
            "burn_in_days": config.burn_in_days,
            "measure_days": config.measure_days,
            "cooldown_days": config.cooldown_days,
            "seed": config.seed,
        }
        for name, config in configs.items()
    ]
)

class_config_df = pd.DataFrame(
    [
        {
            "scenario": name,
            "class_id": class_id,
            "lambda_per_day": params.lambda_per_day,
            "cancel_prob": params.cancel_prob,
            "value": params.value,
            "balk_prob": describe_rule(params.balk_prob),
            "no_show_prob": describe_rule(params.no_show_prob),
        }
        for name, config in configs.items()
        for class_id, params in config.classes.items()
    ]
)

display(global_config_df)
display(class_config_df)

## Run The Configured Scenarios

Each run constructs `ClinicAppointmentSimulation(config)` and calls `.run()`, matching the command-line scripts.

In [ ]:
def run_scenario(config):
    sim = ClinicAppointmentSimulation(config)
    return sim.run()

results = {name: run_scenario(config) for name, config in configs.items()}
list(results)

In [ ]:
def _rate(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def aggregate_metrics_frame(results_by_name):
    return pd.DataFrame(
        aggregate_result_row(result, {"scenario": name})
        for name, result in results_by_name.items()
    )


def class_metrics_frame(results_by_name):
    df = pd.DataFrame(
        row
        for name, result in results_by_name.items()
        for row in class_result_rows(result, {"scenario": name})
    )
    df["offered_rate"] = [_rate(offered, arrivals) for offered, arrivals in zip(df["offered"], df["arrivals"])]
    df["booking_rate"] = [_rate(booked, arrivals) for booked, arrivals in zip(df["booked"], df["arrivals"])]
    df["balk_rate"] = [_rate(balked, arrivals) for balked, arrivals in zip(df["balked"], df["arrivals"])]
    df["no_offer_rate"] = [_rate(no_offer, arrivals) for no_offer, arrivals in zip(df["no_offer"], df["arrivals"])]
    df["cancel_rate_among_booked"] = [_rate(canceled, booked) for canceled, booked in zip(df["canceled"], df["booked"])]
    df["no_show_rate_among_booked"] = [_rate(no_show, booked) for no_show, booked in zip(df["no_show"], df["booked"])]
    return df


def slot_metrics_frame(results_by_name):
    rows = []
    for name, result in results_by_name.items():
        metrics = result.slot_metrics
        total_slots = result.total_slots
        rows.append(
            {
                "scenario": name,
                "booked_slot_rate": _rate(metrics.booked_slots, total_slots),
                "served_slot_rate": _rate(metrics.served_slots, total_slots),
                "no_show_slot_rate": _rate(metrics.no_show_slots, total_slots),
            }
        )
    return pd.DataFrame(rows)


aggregate_metrics_df = aggregate_metrics_frame(results)
class_metrics_df = class_metrics_frame(results)
slot_metrics_df = slot_metrics_frame(results)

display(
    aggregate_metrics_df.style.format(
        {
            "average_utilization": "{:.1%}",
            "overall_percent_serviced": "{:.1%}",
            "mean_accepted_booking_delay": "{:.2f}",
            "mean_offered_booking_delay": "{:.2f}",
            "overall_balking_rate": "{:.1%}",
        }
    )
)

display(
    class_metrics_df.style.format(
        {
            "offered_rate": "{:.1%}",
            "booking_rate": "{:.1%}",
            "balk_rate": "{:.1%}",
            "no_offer_rate": "{:.1%}",
            "cancel_rate_among_booked": "{:.1%}",
            "no_show_rate_among_booked": "{:.1%}",
            "percent_serviced": "{:.1%}",
            "balking_rate": "{:.1%}",
            "slot_utilization": "{:.1%}",
            "mean_accepted_booking_delay": "{:.2f}",
            "mean_offered_booking_delay": "{:.2f}",
        }
    )
)

display(
    slot_metrics_df.style.format(
        {
            "booked_slot_rate": "{:.1%}",
            "served_slot_rate": "{:.1%}",
            "no_show_slot_rate": "{:.1%}",
        }
    )
)

## State Snapshots

The engine records a start-of-day summary state for every measured day after cancellations and before new arrivals. The rows below show `X_i,r`, the number of class `i` patients already scheduled for residual day `r`, on the first measured day.

In [ ]:
def summary_state_to_frame(summary_state):
    horizon = len(next(iter(summary_state.values())))
    return pd.DataFrame.from_dict(
        summary_state,
        orient="index",
        columns=[f"D+{r}" for r in range(horizon)],
    ).rename_axis("class_id")


def full_state_to_frame(result):
    if not result.final_full_state:
        return pd.DataFrame()
    return pd.DataFrame(
        result.final_full_state,
        columns=[f"slot_{j + 1}" for j in range(len(result.final_full_state[0]))],
    ).rename_axis("residual_day")

for name, result in results.items():
    print(f"{name}: first measured-day summary state")
    display(summary_state_to_frame(result.daily_summary_states[0]))

print("Final day-level calendar view after cooldown: Baseline")
display(full_state_to_frame(results["Baseline"]))

## Scenario-Level Outputs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

rate_cols = ["average_utilization", "overall_percent_serviced"]
aggregate_metrics_df.set_index("scenario")[rate_cols].plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Aggregate relative metrics")
axes[0].set_ylabel("Rate")
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].legend(["Average utilization", "Overall percent serviced"], loc="lower right")

aggregate_metrics_df.set_index("scenario")[["mean_accepted_booking_delay", "mean_offered_booking_delay"]].plot(
    kind="bar",
    ax=axes[1],
    rot=0,
)
axes[1].set_title("Aggregate booking-delay metrics")
axes[1].set_ylabel("Delay in days")
axes[1].legend(["Accepted delay", "Offered delay"], loc="upper right")

plt.show()

## Patient Flow By Class

In [ ]:
flow_df = class_metrics_df.copy()
flow_df["scenario_class"] = flow_df["scenario"] + " C" + flow_df["class_id"].astype(str)
flow_df = flow_df.set_index("scenario_class")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)

flow_df[["booking_rate", "balk_rate", "no_offer_rate"]].plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    color=[ARRIVAL_COLOR, BALKING_COLOR, BASELINE_COLOR],
)
axes[0].set_title("Arrival disposition rates")
axes[0].set_ylabel("Share of arrivals")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].legend(loc="upper right")

flow_df[["percent_serviced", "cancel_rate_among_booked", "no_show_rate_among_booked"]].plot(
    kind="bar",
    ax=axes[1],
    color=[ARRIVAL_COLOR, CANCELLATION_COLOR, NO_SHOW_COLOR],
)
axes[1].set_title("Outcome rates")
axes[1].set_ylabel("Rate")
axes[1].set_xlabel("")
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].legend(loc="upper right")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

accepted_delay_df = class_metrics_df.pivot(index="class_id", columns="scenario", values="mean_accepted_booking_delay")
offered_delay_df = class_metrics_df.pivot(index="class_id", columns="scenario", values="mean_offered_booking_delay")

accepted_delay_df.plot(kind="bar", ax=axes[0], rot=0, color=[ARRIVAL_COLOR, CANCELLATION_COLOR])
axes[0].set_title("Mean accepted booking delay by class")
axes[0].set_xlabel("Patient class")
axes[0].set_ylabel("Delay in days")
axes[0].legend(title="Scenario")

offered_delay_df.plot(kind="bar", ax=axes[1], rot=0, color=[ARRIVAL_COLOR, CANCELLATION_COLOR])
axes[1].set_title("Mean offered booking delay by class")
axes[1].set_xlabel("Patient class")
axes[1].set_ylabel("Delay in days")
axes[1].legend(title="Scenario")

plt.show()

## Calendar Load During Measurement

The heatmaps use the recorded start-of-day summary states. Each cell is the total number of patients already scheduled at the start of a measured day for residual day `r`.

In [ ]:
def summary_state_records(results_by_name):
    records = []
    for scenario, result in results_by_name.items():
        for measured_day, summary_state in enumerate(result.daily_summary_states):
            for class_id, counts in summary_state.items():
                for residual_day, booked_count in enumerate(counts):
                    records.append(
                        {
                            "scenario": scenario,
                            "measured_day": measured_day,
                            "class_id": class_id,
                            "residual_day": residual_day,
                            "booked_count": booked_count,
                        }
                    )
    return pd.DataFrame(records)

summary_state_df = summary_state_records(results)
summary_state_df.head()

In [ ]:
scenario_names = list(results)
fig, axes = plt.subplots(1, len(scenario_names), figsize=(6 * len(scenario_names), 4.5), constrained_layout=True, squeeze=False)

for ax, scenario in zip(axes.ravel(), scenario_names):
    heatmap_df = (
        summary_state_df.loc[summary_state_df["scenario"] == scenario]
        .groupby(["measured_day", "residual_day"], as_index=False)["booked_count"]
        .sum()
        .pivot(index="measured_day", columns="residual_day", values="booked_count")
        .sort_index()
    )
    image = ax.imshow(heatmap_df.values, aspect="auto", origin="lower", cmap="Blues")
    ax.set_title(f"{scenario}: booked load")
    ax.set_xlabel("Residual day r")
    ax.set_ylabel("Measured day index")
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns)
    fig.colorbar(image, ax=ax, label="Patients scheduled")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(scenario_names), figsize=(6 * len(scenario_names), 4), constrained_layout=True, squeeze=False)

for ax, scenario in zip(axes.ravel(), scenario_names):
    average_profile = (
        summary_state_df.loc[summary_state_df["scenario"] == scenario]
        .groupby(["residual_day", "class_id"])["booked_count"]
        .mean()
        .unstack("class_id")
    )
    average_profile.plot(marker="o", ax=ax)
    ax.set_title(f"{scenario}: average start-of-day load")
    ax.set_xlabel("Residual day r")
    ax.set_ylabel("Average patients scheduled")
    ax.legend(title="Class")

plt.show()

## Replication Check

A single seed is useful for debugging, but the model is stochastic. This section re-runs each scenario with shared replicate seeds so the plots show run-to-run variability.

In [ ]:
N_REPLICATIONS = 50
REPLICATION_SEED_START = 10_000


def run_replications(configs_by_name, n_replications=N_REPLICATIONS, seed_start=REPLICATION_SEED_START):
    records = []
    for scenario, config in configs_by_name.items():
        for replication in range(n_replications):
            seeded_config = replace(config, seed=seed_start + replication)
            result = ClinicAppointmentSimulation(seeded_config).run()
            records.append(
                {
                    "scenario": scenario,
                    "replication": replication,
                    "seed": seeded_config.seed,
                    **result_metrics_from_result(result),
                }
            )
    return pd.DataFrame(records)

replication_df = run_replications(configs)
replication_summary_df = (
    replication_df.groupby("scenario")
    .agg(
        average_utilization_mean=("average_utilization", "mean"),
        average_utilization_std=("average_utilization", "std"),
        overall_percent_serviced_mean=("overall_percent_serviced", "mean"),
        overall_percent_serviced_std=("overall_percent_serviced", "std"),
        mean_accepted_booking_delay_mean=("mean_accepted_booking_delay", "mean"),
        mean_accepted_booking_delay_std=("mean_accepted_booking_delay", "std"),
        mean_offered_booking_delay_mean=("mean_offered_booking_delay", "mean"),
        mean_offered_booking_delay_std=("mean_offered_booking_delay", "std"),
    )
    .reset_index()
)

display(replication_summary_df.style.format({col: "{:.3f}" for col in replication_summary_df.columns if col != "scenario"}))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
box_metrics = [
    "average_utilization",
    "overall_percent_serviced",
    "mean_accepted_booking_delay",
    "mean_offered_booking_delay",
]
box_titles = [
    "Average utilization",
    "Overall percent serviced",
    "Mean accepted delay",
    "Mean offered delay",
]

for ax, metric, title in zip(axes, box_metrics, box_titles):
    data = [replication_df.loc[replication_df["scenario"] == scenario, metric] for scenario in scenario_names]
    box = ax.boxplot(data, labels=scenario_names, patch_artist=True)
    for patch, color in zip(box["boxes"], [ARRIVAL_COLOR, CANCELLATION_COLOR]):
        patch.set_facecolor(color)
    ax.set_title(title)
    ax.set_xlabel("")
    if metric in {"average_utilization", "overall_percent_serviced"}:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))

plt.show()

## Notes For Further Runs

Change `CONFIG_PATHS` to add or remove YAML scenarios. Change `N_REPLICATIONS` for a faster or more stable replication check.